# MindGuard — Model Comparison Notebook
## Bi-LSTM vs RoBERTa vs Mental-RoBERTa

**Author:** Diana Atieno Opiyo  
**MSc Thesis:** Deep Learning Model on Suicidal Ideation Detection Using Social Media Text Data  
**Institution:** Technical University of Mombasa  

This notebook trains and compares three models on the same annotated dataset:
1. **Bi-LSTM** — deep learning baseline (bidirectional LSTM)
2. **RoBERTa** — general-purpose transformer pre-trained on 58M tweets
3. **Mental-RoBERTa** — domain-specific transformer pre-trained on mental health text

The best model is saved for deployment in the MindGuard Streamlit app.

> **Before running:** Runtime → Change runtime type → **T4 GPU**

In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 1 — Install required libraries
# transformers : Hugging Face library providing RoBERTa and Mental-RoBERTa
# datasets     : efficient data loading utilities from Hugging Face
# scikit-learn : evaluation metrics and train/test splitting
# ─────────────────────────────────────────────────────────────────
!pip install transformers datasets scikit-learn --quiet
print('Libraries installed.')

In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 2 — Import all libraries
# ─────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import torch
import json, os, re, time, pickle, warnings
warnings.filterwarnings('ignore')

# PyTorch — used for RoBERTa and Mental-RoBERTa
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    AutoTokenizer,                          # loads the right tokenizer for any model
    AutoModelForSequenceClassification,     # loads model with classification head
    get_linear_schedule_with_warmup,        # learning rate scheduler
)

# TensorFlow / Keras — used for Bi-LSTM
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding, Bidirectional, LSTM,
    Dense, SpatialDropout1D
)
from tensorflow.keras.preprocessing.text import Tokenizer as KerasTokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# Evaluation metrics
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, accuracy_score
)

# Set device — GPU if available, otherwise CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch device  : {device}')
print(f'TensorFlow GPUs : {tf.config.list_physical_devices("GPU")}')
if torch.cuda.is_available():
    print(f'GPU             : {torch.cuda.get_device_name(0)}')
else:
    print('WARNING: No GPU detected. Training will be slow. Enable T4 GPU in Runtime settings.')

In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 3 — Upload the annotated dataset
# Click 'Choose Files' and upload: suicidal_ideation_reddit_annotated.csv
# ─────────────────────────────────────────────────────────────────
from google.colab import files
uploaded = files.upload()   # upload the CSV file
print('File uploaded successfully.')

In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 4 — Load and explore the dataset
# Expected columns:
#   usertext : the Reddit post text
#   label    : 1 = suicidal ideation detected, 0 = not detected
# ─────────────────────────────────────────────────────────────────
df = pd.read_csv('suicidal_ideation_reddit_annotated.csv')

# Remove rows with missing values
df = df.dropna(subset=['usertext', 'label'])
df['label'] = df['label'].astype(int)

print(f'Dataset loaded successfully.')
print(f'Total samples        : {len(df)}')
print(f'Suicidal     (1)     : {(df["label"]==1).sum()} ({(df["label"]==1).mean()*100:.1f}%)')
print(f'Non-suicidal (0)     : {(df["label"]==0).sum()} ({(df["label"]==0).mean()*100:.1f}%)')
print(f'Avg text length (words): {df["usertext"].str.split().str.len().mean():.0f}')
print()
print('Sample suicidal post (label=1):')
print(df[df['label']==1]['usertext'].iloc[0][:200])
print()
print('Sample non-suicidal post (label=0):')
print(df[df['label']==0]['usertext'].iloc[0][:200])

In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 5 — Split dataset into train, validation, and test sets
#
# All three models use the EXACT SAME splits so the comparison is fair.
#   Train : 75%  — used to update model weights
#   Val   : 10%  — used to monitor training and prevent overfitting
#   Test  : 15%  — held out completely, used ONLY for final evaluation
#
# stratify=labels ensures equal class proportions in every split.
# ─────────────────────────────────────────────────────────────────
RANDOM_SEED = 42   # fixed seed ensures reproducible splits

texts  = df['usertext'].tolist()
labels = df['label'].tolist()

# Step 1: separate test set (15%)
train_val_texts, test_texts, train_val_labels, test_labels = train_test_split(
    texts, labels,
    test_size=0.15,
    random_state=RANDOM_SEED,
    stratify=labels
)

# Step 2: separate validation from training (~10% of total)
train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_val_texts, train_val_labels,
    test_size=0.118,
    random_state=RANDOM_SEED,
    stratify=train_val_labels
)

print(f'Train set : {len(train_texts):,} samples ({len(train_texts)/len(df)*100:.1f}%)')
print(f'Val set   : {len(val_texts):,} samples ({len(val_texts)/len(df)*100:.1f}%)')
print(f'Test set  : {len(test_texts):,} samples ({len(test_texts)/len(df)*100:.1f}%)')
print(f'Total     : {len(train_texts)+len(val_texts)+len(test_texts):,} samples')
print()
print('All three models will be trained and evaluated on these SAME splits.')

---
## Model 1 — Bi-LSTM
The deep learning baseline. Uses bidirectional LSTM layers to read text  
forward and backward. Trained from scratch on this dataset using a Keras tokenizer.  
Same architecture as the original MindGuard model.

In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 6 — Bi-LSTM: text cleaning and tokenization
#
# Bi-LSTM requires text cleaning before tokenization because it has
# no pre-trained knowledge — it learns from scratch on this dataset.
# The Keras tokenizer maps words to integer indices.
# pad_sequences makes all inputs the same length (100 tokens).
# ─────────────────────────────────────────────────────────────────
LSTM_MAX_LEN  = 100     # maximum words per input text
LSTM_VOCAB    = 20000   # maximum vocabulary size
LSTM_EMBED    = 128     # word embedding dimensions
LSTM_UNITS1   = 64      # units in first BiLSTM layer
LSTM_UNITS2   = 32      # units in second BiLSTM layer
LSTM_DROPOUT  = 0.5     # dropout rate to prevent overfitting
LSTM_BATCH    = 64      # samples per training batch
LSTM_EPOCHS   = 15      # max epochs (early stopping will stop earlier)
LSTM_PATIENCE = 3       # stop if val_loss does not improve for 3 epochs

def clean_text(text):
    """Lowercase, remove URLs and non-alphabetic characters."""
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)  # remove URLs
    text = re.sub(r'[^a-z\s]', ' ', text)        # keep only letters
    text = re.sub(r'\s+', ' ', text).strip()      # collapse whitespace
    return text

# Clean text for all splits
train_clean = [clean_text(t) for t in train_texts]
val_clean   = [clean_text(t) for t in val_texts]
test_clean  = [clean_text(t) for t in test_texts]

# Fit tokenizer ONLY on training data (never on val or test — that would leak information)
keras_tok = KerasTokenizer(num_words=LSTM_VOCAB, oov_token='<OOV>')
keras_tok.fit_on_texts(train_clean)

# Convert text to padded integer sequences
X_train_lstm = pad_sequences(keras_tok.texts_to_sequences(train_clean), maxlen=LSTM_MAX_LEN)
X_val_lstm   = pad_sequences(keras_tok.texts_to_sequences(val_clean),   maxlen=LSTM_MAX_LEN)
X_test_lstm  = pad_sequences(keras_tok.texts_to_sequences(test_clean),  maxlen=LSTM_MAX_LEN)

y_train_np = np.array(train_labels)
y_val_np   = np.array(val_labels)
y_test_np  = np.array(test_labels)

print(f'X_train shape : {X_train_lstm.shape}')
print(f'X_val shape   : {X_val_lstm.shape}')
print(f'X_test shape  : {X_test_lstm.shape}')
print(f'Vocabulary size (fitted): {len(keras_tok.word_index):,}')

In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 7 — Bi-LSTM: build and train the model
#
# Architecture:
#   Embedding(20K, 128) -> SpatialDropout(0.2)
#   -> BiLSTM(64, dropout=0.5) -> BiLSTM(32, dropout=0.5)
#   -> Dense(1, sigmoid)
#
# EarlyStopping prevents overfitting by stopping when val_loss
# stops improving. ModelCheckpoint saves the best weights.
# ─────────────────────────────────────────────────────────────────
bilstm_model = Sequential([
    # Embedding: converts integer tokens to dense vectors
    Embedding(LSTM_VOCAB, LSTM_EMBED, input_length=LSTM_MAX_LEN),

    # SpatialDropout: drops entire feature maps — better than regular dropout for sequences
    SpatialDropout1D(0.2),

    # First BiLSTM: reads text forward AND backward, returns sequences
    Bidirectional(LSTM(LSTM_UNITS1, dropout=LSTM_DROPOUT,
                       recurrent_dropout=0.2, return_sequences=True)),

    # Second BiLSTM: reads output of first layer, returns single context vector
    Bidirectional(LSTM(LSTM_UNITS2, dropout=LSTM_DROPOUT, recurrent_dropout=0.2)),

    # Output: sigmoid gives probability between 0 and 1
    Dense(1, activation='sigmoid'),
])

bilstm_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0003),
    loss='binary_crossentropy',
    metrics=['accuracy']
)
bilstm_model.summary()

# Train the model
print('\nTraining Bi-LSTM...')
bilstm_history = bilstm_model.fit(
    X_train_lstm, y_train_np,
    validation_data=(X_val_lstm, y_val_np),
    epochs=LSTM_EPOCHS,
    batch_size=LSTM_BATCH,
    callbacks=[
        EarlyStopping(monitor='val_loss', patience=LSTM_PATIENCE,
                      restore_best_weights=True, verbose=1),
        ModelCheckpoint('bilstm_best.keras', monitor='val_loss',
                        save_best_only=True, verbose=0),
    ],
    verbose=1,
)

In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 8 — Bi-LSTM: evaluate on test set
#
# predict() returns probabilities (0.0 to 1.0).
# We threshold at 0.5 to get binary labels.
# ROC-AUC is the primary metric — it measures discrimination ability
# across all thresholds, making it more informative than accuracy.
# ─────────────────────────────────────────────────────────────────
bilstm_model = tf.keras.models.load_model('bilstm_best.keras')  # load best weights

bilstm_probs = bilstm_model.predict(X_test_lstm, verbose=0).flatten()
bilstm_preds = (bilstm_probs >= 0.5).astype(int)

bilstm_acc = accuracy_score(y_test_np, bilstm_preds)
bilstm_auc = roc_auc_score(y_test_np, bilstm_probs)

print('=' * 60)
print('MODEL 1 — BI-LSTM TEST RESULTS')
print('=' * 60)
print(f'Accuracy : {bilstm_acc:.4f} ({bilstm_acc*100:.2f}%)')
print(f'ROC-AUC  : {bilstm_auc:.4f}')
print()
print(classification_report(y_test_np, bilstm_preds,
      target_names=['Non-Suicidal (0)', 'Suicidal (1)'], digits=4))

---
## Model 2 — RoBERTa (twitter-roberta-base)
General-purpose transformer pre-trained on **58 million tweets**.  
Understands informal social media language but has no specific  
mental health domain knowledge.

In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 9 — Shared transformer utilities
# Both RoBERTa and Mental-RoBERTa use the same Dataset class,
# DataLoader setup, training loop, and evaluation function.
# This avoids code duplication and ensures fair comparison.
# ─────────────────────────────────────────────────────────────────

# Shared hyperparameters for both transformer models
ROB_MAX_LEN = 256   # maximum tokens per input (RoBERTa supports up to 512)
ROB_BATCH   = 16    # reduce to 8 if you get CUDA out-of-memory errors
ROB_EPOCHS  = 4     # 3-4 epochs is standard for fine-tuning transformers
ROB_LR      = 2e-5  # standard learning rate for transformer fine-tuning


class SuicideDataset(Dataset):
    """
    PyTorch Dataset that tokenizes Reddit posts for transformer models.
    Works with any Hugging Face tokenizer (RoBERTa or Mental-RoBERTa).

    Returns for each sample:
      input_ids      : token ID sequence the model reads
      attention_mask : 1 for real tokens, 0 for padding
      label          : 0 or 1
    """
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts     = texts
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding='max_length',  # pad shorter texts to max_len
            truncation=True,       # cut longer texts at max_len
            return_tensors='pt',
        )
        return {
            'input_ids'      : enc['input_ids'].squeeze(0),
            'attention_mask' : enc['attention_mask'].squeeze(0),
            'label'          : torch.tensor(self.labels[idx], dtype=torch.long),
        }


def train_transformer_epoch(model, loader, optimizer, scheduler, device):
    """
    Run one training epoch for a transformer model.
    Updates model weights via backpropagation.
    Returns average loss and accuracy for the epoch.
    """
    model.train()  # enable dropout and batch norm
    total_loss, correct, total = 0, 0, 0
    for i, batch in enumerate(loader):
        ids   = batch['input_ids'].to(device)
        mask  = batch['attention_mask'].to(device)
        lbls  = batch['label'].to(device)

        optimizer.zero_grad()  # clear gradients from previous step

        # Forward pass: compute predictions and loss
        out  = model(input_ids=ids, attention_mask=mask, labels=lbls)
        loss = out.loss

        loss.backward()  # backpropagation: compute gradients

        # Clip gradients — prevents exploding gradients in transformer training
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()   # update weights
        scheduler.step()   # update learning rate

        total_loss += loss.item()
        preds = torch.argmax(out.logits, dim=1)
        correct += (preds == lbls).sum().item()
        total   += lbls.size(0)

        if (i + 1) % 50 == 0:
            print(f'    Batch {i+1}/{len(loader)} — loss: {loss.item():.4f}')

    return total_loss / len(loader), correct / total


def evaluate_transformer(model, loader, device):
    """
    Evaluate a transformer model on validation or test data.
    No weight updates — inference mode only.
    Returns: loss, accuracy, predictions, true labels, probabilities.
    """
    model.eval()  # disable dropout
    total_loss, all_preds, all_labels, all_probs = 0, [], [], []
    with torch.no_grad():  # disable gradient computation for speed and memory
        for batch in loader:
            ids   = batch['input_ids'].to(device)
            mask  = batch['attention_mask'].to(device)
            lbls  = batch['label'].to(device)
            out   = model(input_ids=ids, attention_mask=mask, labels=lbls)
            probs = torch.softmax(out.logits, dim=1)
            preds = torch.argmax(out.logits, dim=1)
            total_loss  += out.loss.item()
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(lbls.cpu().numpy())
            all_probs.extend(probs[:, 1].cpu().numpy())  # prob of suicidal class
    acc = accuracy_score(all_labels, all_preds)
    return total_loss / len(loader), acc, all_preds, all_labels, all_probs


def train_transformer(model_name, train_texts, val_texts, test_texts,
                      train_labels, val_labels, test_labels,
                      max_len, batch_size, epochs, lr, device):
    """
    Full training pipeline for a transformer model.
    Loads model, trains for given epochs, evaluates on test set.
    Returns: model, tokenizer, test_accuracy, test_auc, test_preds, test_probs.
    """
    print(f'Loading tokenizer: {model_name}')
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    # Create PyTorch datasets — transformers use raw text, no cleaning needed
    train_ds = SuicideDataset(train_texts, train_labels, tokenizer, max_len)
    val_ds   = SuicideDataset(val_texts,   val_labels,   tokenizer, max_len)
    test_ds  = SuicideDataset(test_texts,  test_labels,  tokenizer, max_len)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False)

    print(f'Loading model: {model_name}')
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=2,                  # binary classification
        ignore_mismatched_sizes=True,  # needed when adding classification head
    ).to(device)

    # Optimizer and learning rate scheduler
    optimizer     = AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    total_steps   = len(train_loader) * epochs
    warmup_steps  = int(0.1 * total_steps)  # warm up for 10% of steps
    scheduler     = get_linear_schedule_with_warmup(
        optimizer, warmup_steps, total_steps
    )

    best_val_acc, best_state = 0.0, None
    history = []

    for epoch in range(1, epochs + 1):
        print(f'\n  Epoch {epoch}/{epochs}')
        tr_loss, tr_acc = train_transformer_epoch(model, train_loader, optimizer, scheduler, device)
        vl_loss, vl_acc, _, vl_lbls, vl_probs = evaluate_transformer(model, val_loader, device)
        vl_auc = roc_auc_score(vl_lbls, vl_probs)
        history.append({'epoch': epoch, 'train_loss': round(tr_loss,4),
                        'train_acc': round(tr_acc,4), 'val_loss': round(vl_loss,4),
                        'val_acc': round(vl_acc,4), 'val_auc': round(vl_auc,4)})
        print(f'  Train — Loss: {tr_loss:.4f} | Acc: {tr_acc:.4f}')
        print(f'  Val   — Loss: {vl_loss:.4f} | Acc: {vl_acc:.4f} | AUC: {vl_auc:.4f}')

        # Save best weights based on validation accuracy
        if vl_acc > best_val_acc:
            best_val_acc = vl_acc
            best_state   = {k: v.clone() for k, v in model.state_dict().items()}
            print(f'  ✓ Best model saved (val_acc={vl_acc:.4f})')

    # Load best weights and evaluate on test set
    model.load_state_dict(best_state)
    _, te_acc, te_preds, te_lbls, te_probs = evaluate_transformer(model, test_loader, device)
    te_auc = roc_auc_score(te_lbls, te_probs)

    return model, tokenizer, te_acc, te_auc, te_preds, te_lbls, te_probs, history


print('Shared transformer utilities defined.')

In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 10 — Train RoBERTa (twitter-roberta-base)
#
# cardiffnlp/twitter-roberta-base was pre-trained on 58 million tweets.
# It understands informal social media language and abbreviations.
# We fine-tune it on our annotated dataset for suicidal ideation detection.
#
# Expected training time: ~30-35 minutes on T4 GPU
# ─────────────────────────────────────────────────────────────────
print('=' * 60)
print('Training Model 2 — RoBERTa (twitter-roberta-base)')
print('=' * 60)

(
    roberta_model,
    roberta_tokenizer,
    roberta_acc,
    roberta_auc,
    roberta_preds,
    roberta_true,
    roberta_probs,
    roberta_history,
) = train_transformer(
    model_name   = 'cardiffnlp/twitter-roberta-base',
    train_texts  = train_texts,
    val_texts    = val_texts,
    test_texts   = test_texts,
    train_labels = train_labels,
    val_labels   = val_labels,
    test_labels  = test_labels,
    max_len      = ROB_MAX_LEN,
    batch_size   = ROB_BATCH,
    epochs       = ROB_EPOCHS,
    lr           = ROB_LR,
    device       = device,
)

print()
print('=' * 60)
print('MODEL 2 — ROBERTA TEST RESULTS')
print('=' * 60)
print(f'Accuracy : {roberta_acc:.4f} ({roberta_acc*100:.2f}%)')
print(f'ROC-AUC  : {roberta_auc:.4f}')
print()
print(classification_report(roberta_true, roberta_preds,
      target_names=['Non-Suicidal (0)', 'Suicidal (1)'], digits=4))

---
## Model 3 — Mental-RoBERTa (mental/mental-roberta-base)
Domain-specific transformer. Started from RoBERTa then additionally  
pre-trained on mental health text — Reddit posts from r/depression,  
r/SuicideWatch, r/mentalhealth, clinical notes, and psychology papers.  
This is the most specialised model for this exact task.

In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 11 — Train Mental-RoBERTa (mental/mental-roberta-base)
#
# mental/mental-roberta-base was additionally pre-trained on:
#   - Reddit posts from r/depression, r/SuicideWatch, r/mentalhealth
#   - Clinical notes and psychology research papers
# This gives it specialist knowledge of how mental distress is expressed
# in text — directly relevant to this suicidal ideation detection task.
#
# We use the EXACT SAME training function as RoBERTa for fair comparison.
# Expected training time: ~30-35 minutes on T4 GPU
# ─────────────────────────────────────────────────────────────────
print('=' * 60)
print('Training Model 3 — Mental-RoBERTa (mental/mental-roberta-base)')
print('=' * 60)

(
    mental_model,
    mental_tokenizer,
    mental_acc,
    mental_auc,
    mental_preds,
    mental_true,
    mental_probs,
    mental_history,
) = train_transformer(
    model_name   = 'mental/mental-roberta-base',
    train_texts  = train_texts,
    val_texts    = val_texts,
    test_texts   = test_texts,
    train_labels = train_labels,
    val_labels   = val_labels,
    test_labels  = test_labels,
    max_len      = ROB_MAX_LEN,
    batch_size   = ROB_BATCH,
    epochs       = ROB_EPOCHS,
    lr           = ROB_LR,
    device       = device,
)

print()
print('=' * 60)
print('MODEL 3 — MENTAL-ROBERTA TEST RESULTS')
print('=' * 60)
print(f'Accuracy : {mental_acc:.4f} ({mental_acc*100:.2f}%)')
print(f'ROC-AUC  : {mental_auc:.4f}')
print()
print(classification_report(mental_true, mental_preds,
      target_names=['Non-Suicidal (0)', 'Suicidal (1)'], digits=4))

---
## Final Comparison — All Three Models
Side-by-side results on the same held-out test set.

In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 12 — Side-by-side comparison of all three models
#
# All three models were evaluated on the SAME test set so the
# comparison is completely fair.
# ROC-AUC is the primary metric — it is threshold-independent
# and more meaningful than accuracy for clinical applications.
# ─────────────────────────────────────────────────────────────────
results = {
    'Bi-LSTM'         : {'accuracy': bilstm_acc, 'auc': bilstm_auc},
    'RoBERTa'         : {'accuracy': roberta_acc, 'auc': roberta_auc},
    'Mental-RoBERTa'  : {'accuracy': mental_acc,  'auc': mental_auc},
}

print('=' * 60)
print('FINAL MODEL COMPARISON — TEST SET RESULTS')
print('=' * 60)
print(f'{"Model":<22} {"Accuracy":>10} {"ROC-AUC":>10}')
print('-' * 60)
for name, m in results.items():
    print(f'{name:<22} {m["accuracy"]*100:>9.2f}% {m["auc"]:>10.4f}')
print()

# Identify the best model based on ROC-AUC
best_model_name = max(results, key=lambda k: results[k]['auc'])
best_auc        = results[best_model_name]['auc']
print(f'WINNER: {best_model_name}')
print(f'Best ROC-AUC: {best_auc:.4f}')
print()
# Show improvement over Bi-LSTM baseline
baseline_auc = results['Bi-LSTM']['auc']
for name, m in results.items():
    if name != 'Bi-LSTM':
        diff = m['auc'] - baseline_auc
        print(f'{name} vs Bi-LSTM: {diff:+.4f} AUC ({diff*100:+.2f}%)')

---
## Save the Best Model
Save whichever model won for deployment in MindGuard.

In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 13 — Save the best model for MindGuard
#
# If Bi-LSTM wins:
#   Saves lstm_model.keras + tokenizer.pkl
#   Drop these directly into your MindGuard folder — no code changes needed.
#
# If RoBERTa or Mental-RoBERTa wins:
#   Saves model weights (.pt) + tokenizer folder + config (.json)
#   We will update Try_streamlit_app.py to load the new model.
# ─────────────────────────────────────────────────────────────────
import shutil

print(f'Saving best model: {best_model_name}')

if best_model_name == 'Bi-LSTM':
    # Save Keras model — direct drop-in for MindGuard
    bilstm_model.save('lstm_model.keras')
    # Save Keras tokenizer as pickle
    with open('tokenizer.pkl', 'wb') as f:
        pickle.dump(keras_tok, f)
    config_out = {
        'model_type'    : 'bilstm',
        'vocab_size'    : LSTM_VOCAB,
        'max_length'    : LSTM_MAX_LEN,
        'test_accuracy' : round(bilstm_acc, 4),
        'test_auc'      : round(bilstm_auc, 4),
    }
    print('Saved: lstm_model.keras, tokenizer.pkl')

elif best_model_name == 'RoBERTa':
    # Save PyTorch model weights
    torch.save(roberta_model.state_dict(), 'mindguard_best_weights.pt')
    # Save tokenizer
    os.makedirs('mindguard_tokenizer', exist_ok=True)
    roberta_tokenizer.save_pretrained('mindguard_tokenizer')
    config_out = {
        'model_type'    : 'roberta',
        'model_name'    : 'cardiffnlp/twitter-roberta-base',
        'max_length'    : ROB_MAX_LEN,
        'test_accuracy' : round(roberta_acc, 4),
        'test_auc'      : round(roberta_auc, 4),
    }
    print('Saved: mindguard_best_weights.pt, mindguard_tokenizer/')

else:  # Mental-RoBERTa
    # Save PyTorch model weights
    torch.save(mental_model.state_dict(), 'mindguard_best_weights.pt')
    # Save tokenizer
    os.makedirs('mindguard_tokenizer', exist_ok=True)
    mental_tokenizer.save_pretrained('mindguard_tokenizer')
    config_out = {
        'model_type'    : 'mental_roberta',
        'model_name'    : 'mental/mental-roberta-base',
        'max_length'    : ROB_MAX_LEN,
        'test_accuracy' : round(mental_acc, 4),
        'test_auc'      : round(mental_auc, 4),
    }
    print('Saved: mindguard_best_weights.pt, mindguard_tokenizer/')

# Save model config
with open('mindguard_model_config.json', 'w') as f:
    json.dump(config_out, f, indent=2)

# Save training histories for thesis documentation
all_history = {
    'bilstm'         : bilstm_history.history,
    'roberta'        : roberta_history,
    'mental_roberta' : mental_history,
    'comparison'     : results,
    'winner'         : best_model_name,
}
with open('training_history.json', 'w') as f:
    json.dump(all_history, f, indent=2)

print('Saved: mindguard_model_config.json, training_history.json')
print(f'\nBest model ({best_model_name}) saved successfully.')

In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 14 — Quick inference test on sample texts
# Verify the best model makes sensible predictions.
# ─────────────────────────────────────────────────────────────────
samples = [
    ("I feel like I cannot go on anymore. Nobody would miss me.",       1),
    ("Just had an amazing day with my friends. Life is so good!",       0),
    ("I have been thinking about ending it all. The pain is too much.", 1),
    ("Looking forward to my vacation next week!",                       0),
    ("I lost my job and feel so hopeless. I do not know what to do.",   1),
]

print(f'Inference test using best model: {best_model_name}')
print('=' * 65)

if best_model_name == 'Bi-LSTM':
    for text, expected in samples:
        seq  = pad_sequences(keras_tok.texts_to_sequences([clean_text(text)]),
                             maxlen=LSTM_MAX_LEN)
        prob = float(bilstm_model.predict(seq, verbose=0)[0][0])
        pred = int(prob >= 0.5)
        ok   = '✓' if pred == expected else '✗'
        print(f'{ok} {"Suicidal" if pred==1 else "Non-Suicidal":15} ({prob:.1%}) | {text[:50]}...')

else:
    # Use whichever transformer won
    best_m   = roberta_model   if best_model_name == 'RoBERTa' else mental_model
    best_tok = roberta_tokenizer if best_model_name == 'RoBERTa' else mental_tokenizer
    best_m.eval()
    for text, expected in samples:
        enc  = best_tok(text, max_length=ROB_MAX_LEN, padding='max_length',
                        truncation=True, return_tensors='pt')
        with torch.no_grad():
            out  = best_m(input_ids=enc['input_ids'].to(device),
                          attention_mask=enc['attention_mask'].to(device))
            prob = torch.softmax(out.logits, dim=1)[0][1].item()
            pred = int(prob >= 0.5)
        ok = '✓' if pred == expected else '✗'
        print(f'{ok} {"Suicidal" if pred==1 else "Non-Suicidal":15} ({prob:.1%}) | {text[:50]}...')

In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 15 — Download all files from Colab
# Run this last cell to download files to your computer.
# ─────────────────────────────────────────────────────────────────
from google.colab import files

if best_model_name == 'Bi-LSTM':
    files.download('lstm_model.keras')
    files.download('tokenizer.pkl')
else:
    # Zip the tokenizer folder for easy download
    shutil.make_archive('mindguard_tokenizer', 'zip', 'mindguard_tokenizer')
    files.download('mindguard_best_weights.pt')
    files.download('mindguard_tokenizer.zip')

files.download('mindguard_model_config.json')
files.download('training_history.json')

print('Downloads started. Check your browser downloads folder.')
print()
print('After downloading:')
if best_model_name == 'Bi-LSTM':
    print('  1. Copy lstm_model.keras and tokenizer.pkl to your MindGuard folder')
    print('  2. No code changes needed — app loads these automatically')
else:
    print('  1. Extract mindguard_tokenizer.zip to a folder called mindguard_tokenizer')
    print('  2. Copy mindguard_best_weights.pt, mindguard_tokenizer/, and')
    print('     mindguard_model_config.json to your MindGuard folder')
    print('  3. Let me know and I will update Try_streamlit_app.py to load the new model')